In [ ]:
import os

folder = r"D:\Sneha\VR-LLM\phi2_final"
print("Files found:")
for f in os.listdir(folder):
    print(f)

In [ ]:
import os

checkpoint_path = r"D:\Sneha\VR-LLM\phi2_final\checkpoint-2865"
print("Files in latest checkpoint:")
for f in os.listdir(checkpoint_path):
    print(f)

In [2]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [2]:
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
import math
from peft import PeftModel, PeftConfig

base_model_name = "microsoft/phi-2"
adapter_path    = r"D:\Sneha\VR-LLM\phi2_final\checkpoint-2865"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(r"D:\Sneha\VR-LLM\phi2_final")

print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    trust_remote_code=True
).to(device)

print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(base_model, adapter_path)

print("Merging adapter into base model...")
model = model.merge_and_unload()   # merges LoRA weights into base model
model = model.to(device)
print("Model ready")

df = pd.read_csv(r"D:\Sneha\VR-LLM\Dataset\test.csv")
prompts = []
for _, row in df.iterrows():
    prompt = f"Question: {row['Question']}\n===SEP===\n"
    prompts.append(prompt)

# ---------------------------
# Generate responses from model
# ---------------------------
model.eval()
responses = []
max_new_tokens = 200

for prompt in tqdm(prompts, desc="Generating responses"):

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    ).to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = output[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    print("=" * 80)
    print("PROMPT:", prompt)
    print("MODEL RESPONSE:")
    print(response)
    
    responses.append(response)
df['model_response'] = responses
eval_csv = "evaluation.csv"
df.to_csv(eval_csv, index=False)
print(f"Saved evaluation CSV to {eval_csv}")

W0623 20:00:13.992000 10684 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading tokenizer...
Loading base model...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading LoRA adapter...
Merging adapter into base model...
Model ready


Generating responses:   0%|                                                            | 1/717 [00:01<18:24,  1.54s/it]

...

PROMPT: Question: Explain lipid metabolism (β-oxidation).
===SEP===

MODEL RESPONSE:
Equation: triglyceride → glycerol + fatty acids
Answer: Fatty acids undergo β-oxidation to produce acetyl-CoA for energy.


Generating responses: 100%|█████████████████████████████████████████████████████████▉| 716/717 [09:19<00:00,  1.02it/s]

PROMPT: Question: Derive K expression using mole fraction.
===SEP===

MODEL RESPONSE:
Equation: K = (Xc * Xp)^(1/2)
Answer: At equilibrium, partial pressures are related to moles by ideal gas equation. Taking the square root of product gives K = (√(Xc*Xp)). This relation is useful for solving equations involving mole fractions directly.


Generating responses: 100%|██████████████████████████████████████████████████████████| 717/717 [09:20<00:00,  1.28it/s]

PROMPT: Question: Explain ion-electron method.
===SEP===

MODEL RESPONSE:
Equation: R–X + Nu− → product
Answer: Electron attack on halogen forms intermediate which is cleaved to give alkane.
Saved evaluation CSV to evaluation.csv


In [5]:
import math
import torch

questions = df["Question"].tolist()

targets = [
    f"Equation: {eq}\nAnswer: {ans}"
    for eq, ans in zip(df["Equation"], df["Answer"])
]

generated = df["model_response"].tolist()

def compute_perplexity(questions, targets, model, tokenizer):

    model.eval()

    losses = []

    sep = "===SEP==="

    sep_ids = tokenizer(
        sep,
        add_special_tokens=False
    )["input_ids"]

    for question, target in zip(questions, targets):
    
        text = f"Question: {question}\n{sep}\n{target}"
    
        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=512
        ).to(device)

        labels = inputs["input_ids"].clone().detach()
        ids = inputs["input_ids"][0].tolist()

        # Find separator
        start = -1
        for i in range(len(ids) - len(sep_ids) + 1):
            if ids[i:i + len(sep_ids)] == sep_ids:
                start = i
                break

        if start != -1:
            labels[:, :start + len(sep_ids)] = -100

        with torch.no_grad():
            outputs = model(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                labels=labels
            )

        losses.append(outputs.loss.item())

    avg_loss = sum(losses) / len(losses)

    return math.exp(avg_loss)


perplexity = compute_perplexity(
    questions,
    targets,
    model,
    tokenizer
)

print("Mean Perplexity:", perplexity)

Mean Perplexity: 2.6778233830219333


In [6]:
!pip install bert-score

In [7]:
from bert_score import score

print("Calculating BERTScore...")

P, R, F1 = score(
    generated,
    targets,
    lang="en",
    verbose=True
)

print("\n===== BERTScore =====")
print(f"Mean Precision : {P.mean().item():.4f}")
print(f"Mean Recall    : {R.mean().item():.4f}")
print(f"Mean F1 Score  : {F1.mean().item():.4f}")

Calculating BERTScore...


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

C:\Users\User\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\User\.cache\huggingface\hub\models--roberta-large. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/23 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/12 [00:00<?, ?it/s]

done in 2.42 seconds, 296.27 sentences/sec

===== BERTScore =====
Mean Precision : 0.9059
Mean Recall    : 0.9121
Mean F1 Score  : 0.9088


In [2]:
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
import math
from peft import PeftModel, PeftConfig

base_model_name = "google/gemma-3-1b-pt"
adapter_path    = r"D:\Sneha\VR-LLM\gemma3_finetuned\checkpoint-3824"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(r"D:\Sneha\VR-LLM\gemma3_finetuned")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    
print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
).to(device)

print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(base_model, adapter_path)

print("Merging adapter into base model...")
model = model.merge_and_unload()   # merges LoRA weights into base model
model = model.to(device)

# print("Loading LoRA adapter...")
# model = PeftModel.from_pretrained(base_model, adapter_path)

# print("LoRA adapter loaded successfully.")

# # Do NOT merge yet
# model.eval()

# prompt = "<|user|>\nHello\n<|assistant|>\n"

# inputs = tokenizer(prompt, return_tensors="pt").to(device)

# with torch.no_grad():
#     outputs = model(**inputs)

# print("Forward pass with LoRA succeeded.")
print("Model ready")

df = pd.read_csv(r"D:\Sneha\VR-LLM\Dataset\test.csv")
prompts = []
for _, row in df.iterrows():
    prompt = f"<|user|>\n{row['Question']}\n<|assistant|>\n"
    prompts.append(prompt)

# ---------------------------
# Generate responses from model
# ---------------------------
model.eval()
responses = []
max_new_tokens = 200

for prompt in tqdm(prompts, desc="Generating responses"):

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    ).to(device)

    # inputs = tokenizer(
    #     prompt,
    #     return_tensors="pt",
    #     max_length=512,
    #     truncation=True
    # ).to(device)
    
    # with torch.no_grad():
    #     outputs = model(**inputs)
    
    # logits = outputs.logits[:, -1, :]
    
    # print("Contains NaN :", torch.isnan(logits).any().item())
    # print("Contains Inf :", torch.isinf(logits).any().item())
    # print("Max logit    :", logits.max().item())
    # print("Min logit    :", logits.min().item())

    
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = output[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    print("=" * 80)
    print("PROMPT:", prompt)
    print("MODEL RESPONSE:")
    print(response)
    
    responses.append(response)
df['model_response'] = responses
eval_csv = "evaluation.csv"
df.to_csv(eval_csv, index=False)
print(f"Saved evaluation CSV to {eval_csv}")

W0701 20:21:31.287000 11580 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading tokenizer...
Loading base model...
Loading LoRA adapter...
Merging adapter into base model...
Model ready


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
C:\Users\User\anaconda3\Lib\site-packages\torch\_inductor\compile_fx.py:322: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(
Generating responses:   0%|                                                          | 1/717 [00:26<5:12:19, 26.17s/it]

PROMPT: <|user|>
Which has higher dielectric constant: phenol or acetic acid?
<|assistant|>

MODEL RESPONSE:
Phenol has higher dielectric constant due to strong intermolecular forces. In phenols, the positive charge is localized within molecules while in acids it resides at molecular boundaries. As a result, effective permittivity is greater. This contrasts with acetic acid, where no net charge exists and there are no dipole–dipole interactions, leading to lower value of K. Therefore, phenol shows stronger intermolecular attraction than acetic acid. Hence, it exhibits higher relative dielectric constant. The stronger interaction increases resistance to electromagnetic waves. Thus, phenol possesses better insulating properties compared to acetic acid.


Generating responses:   0%|▏                                                         | 2/717 [00:28<2:22:25, 11.95s/it]

...

PROMPT: <|user|>
Explain lipid metabolism (β-oxidation).
<|assistant|>

MODEL RESPONSE:
Triglycerides are hydrolyzed in cytosol producing fatty acids and glycerol. Remaining alcohol portion is oxidized yielding acetyl CoA used in synthesis of amino acids, nucleotides etc. This process occurs in cytoplasm and mitochondria. It ensures efficient storage and conversion.

Cholesterol undergoes non-enzymatic oxidation producing ketone bodies causing ketoacidosis in diabetics. Cholesterol can also be converted into ketone bodies due to mutations. Ketones act as warning signs indicating severe ketosis.

Excessive ketone production leads to acidosis requiring insulin therapy. Mutations affecting cholesterol pathway may cause similar complications. Regular monitoring and timely intervention are crucial for managing these disorders effectively.

Ketone bodies accumulate in blood leading to coma and death if not treated promptly. Therefore early detection and management are essential. Genetic test

Generating responses: 100%|█████████████████████████████████████████████████████████▉| 716/717 [21:21<00:01,  1.84s/it]

PROMPT: <|user|>
Derive K expression using mole fraction.
<|assistant|>

MODEL RESPONSE:
Mole fraction is used to convert mass percent into moles per litre. Rearranging gives molarity. This simplifies mole conversion calculations. Example problems involve converting mass percent into moles. The derived relation helps in efficient calculation. Molality and normality derivations follow similar logic. Mole fraction enables direct conversion of mass percent into stoichiometric quantities. It reduces errors due to unit conversions. Hence, it is highly recommended for systematic mole concentration derivation. Molality and normality derivations are complementary derivations that provide detailed explanations for their respective relationships. Mole fraction directly relates to stoichiometric coefficients in chemical equations. Molality involves limiting reagent considerations, while normality assumes constant volume assumption. These derivations ensure comprehensive understanding of relations

Generating responses: 100%|██████████████████████████████████████████████████████████| 717/717 [21:23<00:00,  1.79s/it]

PROMPT: <|user|>
Explain ion-electron method.
<|assistant|>

MODEL RESPONSE:
Ionization enthalpy is calculated using electron gain enthalpy of cations and dissociation enthalpy of acids. This helps determine acid strength. Examples include HCl, H2SO4, H3PO4. It provides acidity scale. More acidic than water due to +I effect. Source: Organic Chemistry Atlas by John Wiley & Sons. Further details available in chemistry textbooks. Electron gain enthalpy strongly influences acidity. Source: Inorganic Chemistry by Kittel. Ionization enthalpy increases with electron loss. Higher ionization enthalpies indicate stronger anions. Example includes Cl−, HF, H2O. Source: Inorganic Chemistry by Allyn & Bacon. Stronger anions readily form ions. Electron loss lowers ionization energy. Source: Physical Chemistry by Gaulton et al. Larger anions require higher activation energies for formation of stable ions. Hence greater ionization tendency. Source: Physical Chemistry by Greenbaum & Nock. Smaller anions

In [3]:
import math
import torch

questions = df["Question"].tolist()

targets = df["Answer"].tolist()

generated = df["model_response"].tolist()


def compute_perplexity(questions, targets, model, tokenizer):

    model.eval()

    losses = []

    assistant_tag = "<|assistant|>\n"

    assistant_ids = tokenizer(
        assistant_tag,
        add_special_tokens=False
    )["input_ids"]

    for question, target in zip(questions, targets):

        # EXACT training format
        text = f"<|user|>\n{question}\n<|assistant|>\n{target}"

        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=512
        ).to(device)

        labels = inputs["input_ids"].clone().detach()

        ids = inputs["input_ids"][0].tolist()

        # Find <|assistant|>
        start = -1
        for i in range(len(ids) - len(assistant_ids) + 1):
            if ids[i:i+len(assistant_ids)] == assistant_ids:
                start = i
                break

        if start != -1:
            labels[:, :start + len(assistant_ids)] = -100

        with torch.no_grad():
            outputs = model(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                labels=labels
            )

        losses.append(outputs.loss.item())

    avg_loss = sum(losses) / len(losses)

    return math.exp(avg_loss)


perplexity = compute_perplexity(
    questions,
    targets,
    model,
    tokenizer
)

print("Mean Perplexity:", perplexity)

Mean Perplexity: 12.735805175719992


In [4]:
from bert_score import score

print("Calculating BERTScore...")

P, R, F1 = score(
    generated,
    targets,
    lang="en",
    verbose=True
)

print("\n===== BERTScore =====")
print(f"Mean Precision : {P.mean().item():.4f}")
print(f"Mean Recall    : {R.mean().item():.4f}")
print(f"Mean F1 Score  : {F1.mean().item():.4f}")

Calculating BERTScore...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/23 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/12 [00:00<?, ?it/s]

done in 5.57 seconds, 128.63 sentences/sec

===== BERTScore =====
Mean Precision : 0.8045
Mean Recall    : 0.8768
Mean F1 Score  : 0.8388
